# 03 Hypothesis Test: Online Courses and Job Satisfaction

This notebook runs the formal hypothesis test for the capstone.

The broader project explores learning pathways and workforce outcomes. For the hypothesis test, this notebook focuses on one specific learning pathway: whether respondents used **online courses or certifications** as part of learning to code.

This is narrower than the full EDA, but it fits the workforce-development theme and maps well to structured upskilling, professional development, and digital training pipelines.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Load Cleaned Data

In [ ]:
df = pd.read_csv("../data/processed/stackoverflow_2025_project_cleaned.csv")

df.head()

## Confirm the Learning Pathway Flag

The hypothesis test uses `learned_online_courses`, which identifies respondents who selected online courses or certifications in the `LearnCode` field.

The flag is recreated here if needed so the notebook can run even if the processed file is missing that column.

In [ ]:
if "learned_online_courses" not in df.columns:
    df["learned_online_courses"] = df["LearnCode"].str.contains(
        "Online Courses or Certification",
        na=False
    ).astype(int)

df["learned_online_courses"].value_counts(dropna=False)

# Hypotheses

**Research question:** Do respondents who used online courses or certifications as a learning pathway report different average job satisfaction than respondents who did not?

**Null hypothesis (H0):** There is no difference in mean job satisfaction between respondents who used online courses/certifications and respondents who did not.

**Alternative hypothesis (H1):** There is a difference in mean job satisfaction between respondents who used online courses/certifications and respondents who did not.

**Alpha:** 0.05

Because the two groups may have different sample sizes and variances, this notebook uses Welch's t-test. Because `JobSat` is a bounded 0–10 survey scale, the notebook also runs a Mann-Whitney U test as a robustness check.

## Prepare Groups

In [ ]:
online_courses = df.loc[df["learned_online_courses"] == 1, "JobSat"].dropna()
not_online_courses = df.loc[df["learned_online_courses"] == 0, "JobSat"].dropna()

print("n online courses/certifications:", len(online_courses))
print("n not online courses/certifications:", len(not_online_courses))

In [ ]:
group_summary = pd.DataFrame({
    "group": ["Used online courses/certifications", "Did not use online courses/certifications"],
    "count": [len(online_courses), len(not_online_courses)],
    "mean": [online_courses.mean(), not_online_courses.mean()],
    "median": [online_courses.median(), not_online_courses.median()],
    "std": [online_courses.std(ddof=1), not_online_courses.std(ddof=1)]
})

group_summary

## Visual Comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(group_summary["group"], group_summary["mean"])
plt.ylabel("Average Job Satisfaction")
plt.title("Average Job Satisfaction by Online Course/Certification Use")
plt.ylim(0, 10)
plt.xticks(rotation=15, ha="right")
plt.show()

## Assumption Checks

With a large survey dataset, normality tests can reject normality for very small deviations. These checks are useful context, but the practical size of the difference matters too.

In [ ]:
rng = np.random.default_rng(42)

online_sample = (
    rng.choice(online_courses, 5000, replace=False)
    if len(online_courses) > 5000
    else online_courses
)

not_online_sample = (
    rng.choice(not_online_courses, 5000, replace=False)
    if len(not_online_courses) > 5000
    else not_online_courses
)

shapiro_online = stats.shapiro(online_sample)
shapiro_not_online = stats.shapiro(not_online_sample)
levene_result = stats.levene(online_courses, not_online_courses)

print(f"Shapiro-Wilk, online courses group: W={shapiro_online.statistic:.4f}, p={shapiro_online.pvalue:.4g}")
print(f"Shapiro-Wilk, no online courses group: W={shapiro_not_online.statistic:.4f}, p={shapiro_not_online.pvalue:.4g}")
print(f"Levene's test for equal variances: W={levene_result.statistic:.4f}, p={levene_result.pvalue:.4g}")

## Welch's t-test

In [ ]:
alpha = 0.05

t_stat, t_p = stats.ttest_ind(
    online_courses,
    not_online_courses,
    equal_var=False
)

print(f"Welch's t-statistic: {t_stat:.4f}")
print(f"p-value: {t_p:.6f}")

if t_p < alpha:
    print(f"Result: Reject H0 at alpha={alpha}.")
else:
    print(f"Result: Fail to reject H0 at alpha={alpha}.")

## Effect Size

In [ ]:
def cohens_d(a, b):
    n1 = len(a)
    n2 = len(b)
    pooled_std = np.sqrt(
        ((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1))
        / (n1 + n2 - 2)
    )
    return (a.mean() - b.mean()) / pooled_std

d = cohens_d(online_courses, not_online_courses)
print(f"Cohen's d: {d:.4f}")

## Mann-Whitney U Robustness Check

In [ ]:
u_stat, u_p = stats.mannwhitneyu(
    online_courses,
    not_online_courses,
    alternative="two-sided"
)

print(f"Mann-Whitney U statistic: {u_stat:.1f}")
print(f"p-value: {u_p:.6f}")

## Plain-English Interpretation

Write this after running the cells above:

- If the p-value is below 0.05, the analysis found a statistically significant difference in average job satisfaction between respondents who used online courses/certifications and those who did not.
- If the p-value is 0.05 or higher, the analysis did not find enough evidence to conclude that average job satisfaction differs between the two groups.
- Regardless of statistical significance, interpret Cohen's d to decide whether the difference is practically meaningful.

This test does not prove that online courses cause higher or lower job satisfaction. It only tests whether the two groups differ in this observational survey.